In [1]:
# %%
# STEP 1: Setup — import libraries and define file paths

import os
import shutil
import pandas as pd

# File paths
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
baseline_models_dir = "/Users/jakegehrung/Desktop/data_raw/baseline_models"
archetype_models_dir = "/Users/jakegehrung/Models"

# Overwrite toggle
overwrite = True  # Set to False if you want to skip existing folders

print("Paths loaded successfully.")

Paths loaded successfully.


In [2]:
# %%
# STEP 2: Load EPC dataset and inspect key columns

df = pd.read_csv(epc_file)

# Ensure required columns exist
required_cols = ["BUILDING_REFERENCE_NUMBER", "ARCHETYPE"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Preview
print("Dataset loaded. Sample:")
display(df[required_cols].head())

Dataset loaded. Sample:


,BUILDING_REFERENCE_NUMBER,ARCHETYPE
0,1001829067,SemiDetached_2F
1,1001951858,Detached_2F
2,1001829071,EndTerrace_2F
3,1234761001,Detached_2F
4,1001991633,Detached_1F


In [3]:
# %%
# STEP 3: Clean data (important for folder matching)

df["BUILDING_REFERENCE_NUMBER"] = df["BUILDING_REFERENCE_NUMBER"].astype(str).str.strip()
df["ARCHETYPE"] = df["ARCHETYPE"].astype(str).str.strip()

print("Data cleaned.")

Data cleaned.


In [4]:
# %%
# STEP 4: Main loop — copy archetype model into each building folder

success_count = 0
fail_count = 0
skip_count = 0

for _, row in df.iterrows():
    building_id = row["BUILDING_REFERENCE_NUMBER"]
    archetype = row["ARCHETYPE"]
    
    # Paths
    building_folder = os.path.join(baseline_models_dir, building_id)
    archetype_folder = os.path.join(archetype_models_dir, archetype)
    destination_folder = os.path.join(building_folder, archetype)
    
    # Debug info
    print(f"\nProcessing Building ID: {building_id}")
    print(f"Archetype: {archetype}")
    
    # Check if building folder exists
    if not os.path.exists(building_folder):
        print(f"❌ Building folder not found: {building_folder}")
        fail_count += 1
        continue
    
    # Check if archetype folder exists
    if not os.path.exists(archetype_folder):
        print(f"❌ Archetype folder not found: {archetype_folder}")
        fail_count += 1
        continue
    
    # Handle overwrite logic
    if os.path.exists(destination_folder):
        if overwrite:
            try:
                shutil.rmtree(destination_folder)
                print(f"♻️ Removed existing folder: {destination_folder}")
            except Exception as e:
                print(f"❌ Error removing existing folder: {e}")
                fail_count += 1
                continue
        else:
            print(f"⚠️ Destination exists, skipping: {destination_folder}")
            skip_count += 1
            continue
    
    # Copy folder
    try:
        shutil.copytree(archetype_folder, destination_folder)
        print(f"✅ Copied to: {destination_folder}")
        success_count += 1
    except Exception as e:
        print(f"❌ Error copying: {e}")
        fail_count += 1

print("\n--- SUMMARY ---")
print(f"Successful copies: {success_count}")
print(f"Failures: {fail_count}")
print(f"Skipped: {skip_count}")


Processing Building ID: 1001829067
Archetype: SemiDetached_2F
♻️ Removed existing folder: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F
✅ Copied to: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F

Processing Building ID: 1001951858
Archetype: Detached_2F
♻️ Removed existing folder: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/Detached_2F
✅ Copied to: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/Detached_2F

Processing Building ID: 1001829071
Archetype: EndTerrace_2F
♻️ Removed existing folder: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829071/EndTerrace_2F
✅ Copied to: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829071/EndTerrace_2F

Processing Building ID: 1234761001
Archetype: Detached_2F
♻️ Removed existing folder: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761001/Detached_2F
✅ Copied to: /Users/jakegehrung/Desktop/data_raw/baseline_models/12347